# 05 — Reinforcement Learning: PPO Training
**Goal**: Train the SFT model using Proximal Policy Optimization (PPO) with execution-guided sandbox rewards and KL divergence penalties (KL $\beta=0.02$, LR $1\times 10^{-6}$). Produces `./checkpoints/ppo/final`.

---

## Step 1: Environment & Tokenizer Configuration

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training
from src.utils.checkpoint import auto_checkpoint, guard_session_limit

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Load APPS Dataset & Tokenizer

In [ ]:
SFT_CHECKPOINT = "./checkpoints/sft/final"
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading APPS training dataset for PPO...")
apps = load_dataset('codeparrot/apps', split='train[:1000]', trust_remote_code=True)
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)
print(f"Prepared {len(apps_clean)} APPS problems for PPO rollout.")

## Step 3: Run PPO Training with Auto-Checkpointing
Integrates background auto-checkpointing every 30 minutes and an 11.5-hour forced save session guard.

In [ ]:
session_start = time.time()
print("Starting PPO Reinforcement Learning Training loop...")

ppo_trainer = run_ppo_training(
    sft_model_path=SFT_CHECKPOINT if os.path.exists(SFT_CHECKPOINT) else MODEL_NAME,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir="./checkpoints/ppo",
    num_epochs=10,
    learning_rate=1e-6,
    batch_size=16,
    mini_batch_size=4,
    gradient_accumulation_steps=4,
    init_kl_coef=0.02,
    target_kl=6.0,
)

print("\nPPO Training completed successfully!")
print("Saved final PPO adapter checkpoint to ./checkpoints/ppo/final")